<a href="https://colab.research.google.com/github/mujocolab/mjlab/blob/main/notebooks/CN/create_new_task.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="在 Colab 中打开"/></a>


# **🤖 mjlab CartPole 教程**

本 Notebook 演示如何使用 mjlab 创建自定义强化学习任务。我们将从零开始构建一个 CartPole（倒立摆）环境，包括：

1. **机器人定义**——用 MuJoCo XML 定义 CartPole 模型
2. **任务配置**——设置观测、动作、奖励与终止条件
3. **训练**——使用 PPO 训练策略
4. **评估**——用训练好的策略可视化仿真

> **说明**：本教程演示如何在 mjlab 中创建新任务。更多背景信息请参阅[官方文档](https://mujocolab.github.io/mjlab/)。

> 本文档翻译自英文原版 Notebook `notebooks/create_new_task.ipynb`。


## **📦 环境搭建与安装**


In [ ]:
# 克隆 mjlab 仓库
!if [ ! -d "mjlab" ]; then git clone -q https://github.com/mujocolab/mjlab.git; fi
%cd /content/mjlab

# 以可编辑模式安装 mjlab
!pip install -e . -q

print("✓ Installation complete!")

### **🔑 WandB 配置**

配置 Weights & Biases 以跟踪实验。在 Colab Secrets 中添加你的 WandB API 密钥：
- `WANDB_API_KEY`：从 [wandb.ai/authorize](https://wandb.ai/authorize) 获取
- `WANDB_ENTITY`：你的 wandb 实体名称


In [ ]:
import os

from google.colab import userdata

try:
  # 设置此项以启用 wandb 日志记录
  os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
  os.environ["WANDB_ENTITY"] = userdata.get("WANDB_ENTITY")

  print("✓ WandB configured successfully!")
except (AttributeError, KeyError):
  # 设置此项以禁用 wandb 日志记录
  os.environ["WANDB_MODE"] = "disabled"

  print("⚠ WandB secrets not found. Training will proceed without WandB logging.")

---

## **🤖 第 1 步：定义机器人**

我们将创建一个简单的 CartPole 机器人，包含：
- 一个滑动的小车（1 个自由度）
- 一根铰链连接的杆（1 个自由度）
- 一个用于控制小车的速度执行器


### **📁 创建目录结构**


In [ ]:
# 创建 cartpole 机器人的目录结构
!mkdir -p /content/mjlab/src/mjlab/asset_zoo/robots/cartpole/
!mkdir -p /content/mjlab/src/mjlab/asset_zoo/robots/cartpole/xmls

print("✓ Directory structure created")

### **📝 创建 MuJoCo XML 模型**

该 XML 定义了 CartPole 的物理模型：
- **地面平面**：用于可视化
- **小车刚体**：带有滑动关节（范围 ±2 米）
- **杆刚体**：带有铰链关节（范围 ±90°）
- **速度执行器**：用于控制小车


In [ ]:
%%writefile /content/mjlab/src/mjlab/asset_zoo/robots/cartpole/xmls/cartpole.xml
<mujoco model="cartpole">
  <compiler angle="degree" coordinate="local" inertiafromgeom="true"/>
  <worldbody>
    <geom name="ground" type="plane" pos="0 0 0" size="5 5 0.1" rgba="0.8 0.9 0.8 1"/>
    <body name="cart" pos="0 0 0.1">
      <geom type="box" size="0.2 0.1 0.1" rgba="0.2 0.2 0.8 1" mass="1.0"/>
      <joint name="slide" type="slide" axis="1 0 0" limited="true" range="-2 2"/>
      <body name="pole" pos="0 0 0.1">
        <geom type="capsule" size="0.05 0.5" fromto="0 0 0 0 0 1" rgba="0.8 0.2 0.2 1" mass="2.0"/>
        <joint name="hinge" type="hinge" axis="0 1 0" range="-90 90"/>
      </body>
    </body>
  </worldbody>
  <actuator>
    <velocity name="slide_velocity" joint="slide" ctrlrange="-20 20" kv="20"/>
  </actuator>
</mujoco>

### **⚙️ 创建机器人配置**


In [ ]:
%%writefile /content/mjlab/src/mjlab/asset_zoo/robots/cartpole/cartpole_constants.py
from pathlib import Path
import mujoco

from mjlab import MJLAB_SRC_PATH
from mjlab.entity import Entity, EntityCfg, EntityArticulationInfoCfg
from mjlab.actuator import XmlActuatorCfg

CARTPOLE_XML: Path = (
  MJLAB_SRC_PATH / "asset_zoo" / "robots" / "cartpole" / "xmls" / "cartpole.xml"
)
assert CARTPOLE_XML.exists(), f"XML not found: {CARTPOLE_XML}"

def get_spec() -> mujoco.MjSpec:
  return mujoco.MjSpec.from_file(str(CARTPOLE_XML))

def get_cartpole_robot_cfg() -> EntityCfg:
  """获取一个新的 CartPole 机器人配置实例。"""
  actuators = (
    XmlActuatorCfg(
      target_names_expr=("slide",),
    ),
  )
  articulation = EntityArticulationInfoCfg(actuators=actuators)
  return EntityCfg(
    spec_fn=get_spec,
    articulation=articulation
  )

# if __name__ == "__main__":
#   import mujoco.viewer as viewer
#   robot = Entity(get_cartpole_robot_cfg())
#   viewer.launch(robot.spec.compile())

In [ ]:
# Create __init__.py for the cartpole robot package
%%writefile /content/mjlab/src/mjlab/asset_zoo/robots/cartpole/__init__.py
# 空的 __init__.py，用于将该目录标记为 Python 包

In [ ]:
import sys

# 将 src 目录加入 Python 路径
mjlab_src = "/content/mjlab/src"
if mjlab_src not in sys.path:
  sys.path.insert(0, mjlab_src)
  print(f"✓ Added {mjlab_src} to Python path")

### **✅ 验证机器人配置**

我们来测试一下机器人能否被正确加载。


In [ ]:
from mjlab.asset_zoo.robots.cartpole.cartpole_constants import get_cartpole_robot_cfg

from mjlab.entity import Entity

# 加载机器人
robot = Entity(get_cartpole_robot_cfg())
model = robot.spec.compile()

# 显示机器人信息
print("✓ CartPole robot loaded successfully!")
print(f"  • Degrees of Freedom (DOF): {model.nv}")
print(f"  • Number of Actuators: {model.nu}")
print(f"  • Bodies: {model.nbody}")
print(f"  • Joints: {model.njnt}")

### **📋 注册机器人**

将 CartPole 机器人添加到 asset zoo 注册表。


In [ ]:
# 将 CartPole 的导入添加到 robots 的 __init__.py
with open("/content/mjlab/src/mjlab/asset_zoo/robots/__init__.py", "a") as f:
  f.write("\n# CartPole robot\n")
  f.write("from mjlab.asset_zoo.robots.cartpole.cartpole_constants import ")
  f.write("get_cartpole_robot_cfg as get_cartpole_robot_cfg\n")

print("✓ CartPole robot registered in asset zoo")

---

## **🎯 第 2 步：定义任务（MDP）**

现在我们来定义马尔可夫决策过程：
- **观测**：杆的角度、角速度，小车的位置、速度
- **动作**：小车速度指令
- **奖励**：直立奖励 + 能耗惩罚
- **终止**：杆倾倒或超时
- **事件**：随机推力，用于提升鲁棒性


### **📁 创建任务目录**


In [ ]:
!mkdir -p /content/mjlab/src/mjlab/tasks/cartpole

print("✓ Task directory created")

### **📝 创建环境配置**

该文件包含 MDP（马尔可夫决策过程）的各个组成部分：
1. **场景配置**：64 个并行环境
2. **动作**：关节速度控制，缩放系数为 20.0
3. **观测**：归一化后的状态变量
4. **奖励**：直立奖励（5.0）+ 能耗惩罚（-0.01）
5. **事件**：关节重置 + 随机推力
6. **终止**：杆倾倒（>30°）或超时（10 秒）


In [ ]:
%%writefile /content/mjlab/src/mjlab/tasks/cartpole/env_cfg.py
"""CartPole 任务环境配置。"""

import math
import torch

from mjlab.envs import ManagerBasedRlEnvCfg
from mjlab.envs.mdp.actions import JointVelocityActionCfg
from mjlab.managers.observation_manager import ObservationGroupCfg, ObservationTermCfg
from mjlab.managers.reward_manager import RewardTermCfg
from mjlab.managers.termination_manager import TerminationTermCfg
from mjlab.managers.event_manager import EventTermCfg
from mjlab.managers.scene_entity_config import SceneEntityCfg
from mjlab.scene import SceneCfg
from mjlab.sim import MujocoCfg, SimulationCfg
from mjlab.viewer import ViewerConfig
from mjlab.asset_zoo.robots.cartpole.cartpole_constants import get_cartpole_robot_cfg
from mjlab.envs import mdp


def cartpole_env_cfg(play: bool = False) -> ManagerBasedRlEnvCfg:
  """创建 CartPole 环境配置。

  参数：
    play: 若为 True，则禁用观测噪声（corruption）并延长回合长度，用于评估。
  """

  # ==============================================================================
  # 场景配置
  # ==============================================================================

  scene_cfg = SceneCfg(
    num_envs=64 if not play else 16,  # play 模式下使用更少的环境
    extent=1.0,   # 环境之间的间距
    entities={"robot": get_cartpole_robot_cfg()},
  )

  viewer_cfg = ViewerConfig(
    origin_type=ViewerConfig.OriginType.ASSET_BODY,
    entity_name="robot",
    body_name="pole",
    distance=3.0,
    elevation=10.0,
    azimuth=90.0,
  )

  sim_cfg = SimulationCfg(
    mujoco=MujocoCfg(
      timestep=0.02,  # 50 Hz 控制频率
      iterations=1,
    ),
  )

  # ==============================================================================
  # 动作
  # ==============================================================================

  actions = {
    "joint_pos": JointVelocityActionCfg(
      entity_name="robot",
      actuator_names=(".*",),
      scale=20.0,
      use_default_offset=False,
    ),
  }

  # ==============================================================================
  # 观测
  # ==============================================================================

  actor_terms = {
    "angle": ObservationTermCfg(
      func=lambda env: env.sim.data.qpos[:, 1:2] / math.pi
    ),
    "ang_vel": ObservationTermCfg(
      func=lambda env: env.sim.data.qvel[:, 1:2] / 5.0
    ),
    "cart_pos": ObservationTermCfg(
      func=lambda env: env.sim.data.qpos[:, 0:1] / 2.0
    ),
    "cart_vel": ObservationTermCfg(
      func=lambda env: env.sim.data.qvel[:, 0:1] / 20.0
    ),
  }

  observations = {
    "actor": ObservationGroupCfg(
      terms=actor_terms,
      concatenate_terms=True,
      enable_corruption=not play,  # 在 play 模式下禁用观测噪声
    ),
    "critic": ObservationGroupCfg(
      terms=actor_terms,  # Critic 使用相同的观测
      concatenate_terms=True,
      enable_corruption=False,
    ),
  }

  # ==============================================================================
  # 奖励
  # ==============================================================================

  def compute_upright_reward(env):
    """保持杆直立的奖励（角度的余弦值）。"""
    return env.sim.data.qpos[:, 1].cos()

  def compute_effort_penalty(env):
    """控制能耗惩罚。"""
    return -0.01 * (env.sim.data.ctrl[:, 0] ** 2)

  rewards = {
    "upright": RewardTermCfg(func=compute_upright_reward, weight=5.0),
    "effort": RewardTermCfg(func=compute_effort_penalty, weight=1.0),
  }

  # ==============================================================================
  # 事件
  # ==============================================================================

  def random_push_cart(env, env_ids, force_range=(-5, 5)):
    """为小车施加随机力，用于鲁棒性训练。"""
    n = len(env_ids)
    random_forces = (
      torch.rand(n, device=env.device) *
      (force_range[1] - force_range[0]) +
      force_range[0]
    )
    env.sim.data.qfrc_applied[env_ids, 0] = random_forces

  events = {
    "reset_robot_joints": EventTermCfg(
      func=mdp.reset_joints_by_offset,
      mode="reset",
      params={
        "asset_cfg": SceneEntityCfg("robot"),
        "position_range": (-0.1, 0.1),
        "velocity_range": (-0.1, 0.1),
      },
    ),
  }

  # 仅在训练模式下添加随机推力
  if not play:
    events["random_push"] = EventTermCfg(
      func=random_push_cart,
      mode="interval",
      interval_range_s=(1.0, 2.0),
      params={"force_range": (-20.0, 20.0)},
    )

  # ==============================================================================
  # 终止条件
  # ==============================================================================

  def check_pole_tipped(env):
    """检查杆是否倾斜超过 30 度。"""
    return env.sim.data.qpos[:, 1].abs() > math.radians(30)

  terminations = {
    "timeout": TerminationTermCfg(func=mdp.time_out, time_out=True),
    "tipped": TerminationTermCfg(func=check_pole_tipped, time_out=False),
  }

  # ==============================================================================
  # 环境配置
  # ==============================================================================

  return ManagerBasedRlEnvCfg(
    scene=scene_cfg,
    observations=observations,
    actions=actions,
    rewards=rewards,
    events=events,
    terminations=terminations,
    sim=sim_cfg,
    viewer=viewer_cfg,
    decimation=1,           # 不重复执行动作
    episode_length_s=int(1e9) if play else 10.0,  # play 模式为无限长，训练模式为 10 秒
  )

### **⚙️ 创建 RL 配置**

该文件定义了 PPO（近端策略优化）的训练参数。


In [ ]:
%%writefile /content/mjlab/src/mjlab/tasks/cartpole/rl_cfg.py
"""CartPole 任务的 RL 配置。"""

from mjlab.rl.config import (
  RslRlOnPolicyRunnerCfg,
  RslRlModelCfg,
  RslRlPpoAlgorithmCfg,
)


def cartpole_ppo_runner_cfg() -> RslRlOnPolicyRunnerCfg:
  """为 CartPole 任务创建 RL 运行器配置。"""
  return RslRlOnPolicyRunnerCfg(
    actor=RslRlModelCfg(
      hidden_dims=(256, 128, 64),  # 任务较简单，使用更小的网络
      activation="elu",
      obs_normalization=True,
      distribution_cfg={
        "class_name": "GaussianDistribution",
        "init_std": 1.0,
        "std_type": "scalar",
      },
    ),
    critic=RslRlModelCfg(
      hidden_dims=(256, 128, 64),
      activation="elu",
      obs_normalization=True,
    ),
    algorithm=RslRlPpoAlgorithmCfg(
      value_loss_coef=1.0,
      use_clipped_value_loss=True,
      clip_param=0.2,
      entropy_coef=0.01,
      num_learning_epochs=5,
      num_mini_batches=4,
      learning_rate=1.0e-3,
      schedule="adaptive",
      gamma=0.99,
      lam=0.95,
      desired_kl=0.01,
      max_grad_norm=1.0,
    ),
    experiment_name="cartpole",
    save_interval=50,
    num_steps_per_env=24,
    max_iterations=5_000,  # 任务较简单，使用更少的迭代次数
  )

### **📋 注册任务环境**

在 mjlab 注册表中注册 CartPole 任务。


In [ ]:
%%writefile /content/mjlab/src/mjlab/tasks/cartpole/__init__.py
"""CartPole 任务注册。"""

from mjlab.tasks.registry import register_mjlab_task
from mjlab.rl.runner import MjlabOnPolicyRunner

from .env_cfg import cartpole_env_cfg
from .rl_cfg import cartpole_ppo_runner_cfg

register_mjlab_task(
  task_id="Mjlab-Cartpole",
  env_cfg=cartpole_env_cfg(),
  play_env_cfg=cartpole_env_cfg(play=True),
  rl_cfg=cartpole_ppo_runner_cfg(),
  runner_cls=MjlabOnPolicyRunner,
)

---

## **🚀 第 3 步：训练智能体**

现在我们来训练一个 PPO 策略，让 CartPole 保持平衡！


In [ ]:
!python -m mjlab.scripts.train Mjlab-Cartpole --agent.max-iterations 201 --agent.save-interval 20

### **📁 查找训练检查点**

训练完成后，检查点会保存在本地。


In [ ]:
import os
import re
from pathlib import Path

# 查找最近一次训练运行
log_dir = Path("/content/mjlab/logs/rsl_rl/cartpole")
if log_dir.exists():
  runs = sorted(log_dir.glob("*"), key=os.path.getmtime, reverse=True)
  if runs:
    latest_run = runs[0]
    print(f"✓ Latest training run: {latest_run.name}\n")

    # 列出检查点——按迭代次数排序
    checkpoints = list(latest_run.glob("model_*.pt"))
    if checkpoints:
      # 提取迭代次数并按数值排序
      def get_iteration(ckpt):
        match = re.search(r"model_(\d+)\.pt", ckpt.name)
        return int(match.group(1)) if match else 0

      checkpoints = sorted(checkpoints, key=get_iteration)

      print(f"Found {len(checkpoints)} checkpoints:")
      for ckpt in checkpoints[-5:]:  # 显示最近 5 个
        size_mb = ckpt.stat().st_size / (1024 * 1024)
        iteration = get_iteration(ckpt)
        print(f"  • {ckpt.name} (iteration {iteration}, {size_mb:.2f} MB)")

      # 保存最后一个检查点的路径
      last_checkpoint = str(checkpoints[-1])
      print(f"\n💾 Last checkpoint: {last_checkpoint}")
    else:
      print("⚠ No checkpoints found yet")
  else:
    print("⚠ No training runs found")
else:
  print("⚠ Log directory not found. Have you run training yet?")

---

## **🎮 第 4 步：可视化训练好的策略**

让我们看看训练好的策略实际运行的效果！


### **🌐 启动 Viser API**


In [ ]:
import subprocess
import sys

process = subprocess.Popen(
  [
    "python",
    "-m",
    "mjlab.scripts.play",
    "Mjlab-Cartpole",
    "--checkpoint_file",
    last_checkpoint,
    "--num_envs",
    "4",
  ],
  stdout=subprocess.PIPE,
  stderr=subprocess.STDOUT,
  universal_newlines=True,
  bufsize=1,
)

detected_port = None

for line in process.stdout:
  print(line, end="")
  sys.stdout.flush()

  # 从 viser 输出中提取端口号
  port_match = re.search(r":(\d{4})", line)
  if port_match and "viser" in line.lower():
    detected_port = int(port_match.group(1))
    print("\n" + "=" * 34)
    print(f"✅ Server is running on port {detected_port}!")
    print("=" * 34)
    break

### **🖥️ 以 iframe 嵌入客户端**


In [ ]:
from google.colab import output

port = detected_port if detected_port else 8081
output.serve_kernel_port_as_iframe(port=port, height=700)